In [13]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [14]:
from datasets import load_dataset

ds = load_dataset("rjac/e-commerce-customer-support-qa")
df = pd.DataFrame(ds['train'])

In [15]:
import pandas as pd

data={}
for i in ds['train']:
    for key,value in i.items():
        if key not in data:
            data[key]=[value]
        else:
            data[key].append(value)
df = pd.DataFrame(data)

print(df.columns)

Index(['issue_area', 'issue_category', 'issue_sub_category',
       'issue_category_sub_category', 'customer_sentiment', 'product_category',
       'product_sub_category', 'issue_complexity', 'agent_experience_level',
       'agent_experience_level_desc', 'conversation', 'qa'],
      dtype='str')


In [ ]:

def extract_customer_text(conv):
    lines = str(conv).split("\n")
    
    customer_lines = []
    for line in lines:
        if line.strip().startswith("Customer:"):
            # remove "Customer:" prefix
            text = line.replace("Customer:", "").strip()
            customer_lines.append(text) 
    return " ".join(customer_lines)
df["text"] = df["conversation"].apply(extract_customer_text)

In [10]:
print(df['text'][0])

Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'm unable to proceed as it's asking for mobile number or email verification. Can you help me with that? My registered mobile number is +1 123-456-7890. Oh, I'm sorry. I might have registered with a different number. Can you please check with my email address instead? It's johndoe@email.com. Okay, I received the code. What do I do with it? Okay, I entered the code, and it's verified now. Thank you for your help. No, that's all. Thank you.


In [ ]:
import re
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df["text"] = df["text"].apply(clean_text)

In [12]:
y = df["issue_category"]
X_text = df["text"]

In [17]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1,2),
    min_df=2    
)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [19]:
model = LinearSVC()
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.85

Classification Report:

                                                         precision    recall  f1-score   support

                             Accessing Warranty Details       1.00      1.00      1.00         4
                  Account Reactivation and Deactivation       0.00      0.00      0.00         2
                                   Account and Shopping       1.00      1.00      1.00         1
                Adding and Changing Account Information       0.83      1.00      0.91         5
                Availability of Faster Delivery Options       1.00      0.67      0.80         3
                             Book Pricing Discrepancies       1.00      1.00      1.00         5
                         Cash on Delivery (CoD) Refunds       0.57      0.80      0.67         5
Contacting Seller's Partnered Courier Service Providers       1.00      1.00      1.00         1
                                       Delivery Process       1.00      1.00      1.00

d:\sourcesys\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\sourcesys\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\sourcesys\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [21]:
y.unique()

<ArrowStringArray>
[                   'Mobile Number and Email Verification',
                                     'Pickup and Shipping',
                          'Replacement and Return Process',
                         'Login Issues and Error Messages',
                                   'Order Delivery Issues',
                   'Account Reactivation and Deactivation',
                          'Cash on Delivery (CoD) Refunds',
                         'Product Availability and Status',
                                    'Product Installation',
                                      'Order Cancellation',
                           'Lost or Missing Warranty Card',
                                     'Return and Exchange',
                                  'Start Date of Warranty',
                                     'Invoice and Payment',
                                    'Account and Shopping',
                                           'Miscellaneous',
                     

In [ ]:
def predict_issue(conversation_text):
    lines = conversation_text.split("\n")
    customer_lines = [
        line.replace("Customer:", "").strip()
        for line in lines if line.strip().startswith("Customer:")
    ]

    text = " ".join(customer_lines)
    text = clean_text(text)
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

In [ ]:
sample_input = """
Customer: I am unable to login to my account.
Customer: It keeps asking for verification code.
"""
print("\nPredicted Issue:", predict_issue(sample_input))


Predicted Issue: Mobile Number and Email Verification
